# Milestone 1 Lab, notebook 6 of 6: Verify and go to production

## Where we are

| Notebook | What you do |
|---|---|
| 1. The business question | Read the brief, explore the data, and answer the question by hand |
| 2. Your first LLM calls | Call a model, get a structured reply, and have AI draft the metric definitions |
| 3. Ask the LLM with all the data | Put the data in the prompt, ask the question, and check the answer |
| 4. What is a tool, what is an agent | Build one tool, run the agent loop by hand, then let LangChain run it |
| 5. The agent answers the question | Give an agent six tools and compare it with the single call |
| **6. Verify and go to production** (you are here) | Check an answer with code, and see the production version |

Model requests in this notebook: **0 (about 15 if you turn on the evaluation)**. OpenRouter's free models allow about 50 a day.

## What this notebook is for

Two things remain.

1. **Verification.** We checked the agent against an answer key. But a real leader asks questions nobody prepared a key for. We need a check that works on any answer.
2. **Production.** Everything so far is brainstorming code in notebooks. The same logic has been rebuilt as tested production code. We'll see how the two connect.

This notebook makes no model requests unless you switch on the evaluation at the end.

In [ ]:
import sys

import lab_helpers
import pandas as pd
from lab_helpers import LAST_MONTH, MILESTONE, PREVIOUS_MONTH, load_result, show_doc

# Let this notebook import the production package, which is in this milestone's folder
sys.path.insert(0, str(MILESTONE))

from retail_profit_stack import tools as production_tools
from retail_profit_stack.agent import run_analyst
from retail_profit_stack.evaluation import load_cases, run_evaluation, summarise
from retail_profit_stack.schemas import AnalystAnswer, ToolCall
from retail_profit_stack.verify import verify_answer

## Step 1: A check that needs no answer key

Load the agent's answer and its tool log from notebook 5.

In [ ]:
agent_answer = AnalystAnswer(**load_result("agent_answer", required_from="notebook 5"))
tool_log = [ToolCall(**call) for call in load_result("agent_tool_log", required_from="notebook 5")]

print(f"The answer has {len(agent_answer.figures)} figures. The agent made {len(tool_log)} tool calls.")

Here is the idea. The tool log records every number the tools returned during the run. So for each figure in the answer we can ask: **did a tool return this exact number, for this metric, month, and product or category?**

If yes, the figure wasn't invented. If no, it was, and we catch it. No answer key needed.

That is the first of seven checks in the **verifier**. The LLD lists them all.

In [ ]:
show_doc("LLD.md", "8. Verifier checks")

## Step 2: Run the verifier

The verifier is plain Python. It calls no model. It lives in the production package, in `retail_profit_stack/verify.py`.

In [ ]:
report = verify_answer(agent_answer, tool_log)

print("Verified:", report.passed)
pd.DataFrame([check.model_dump() for check in report.checks])

If a check failed, read its `detail`. With a free model that's quite possible, and it's the point: the verifier tells you plainly what is wrong with an answer, so a weak answer never reaches the leader looking like a good one.

## Step 3: Try to fool it

Let's play the part of a model that makes up a number. We take the agent's answer and quietly add 100 to one figure.

In [ ]:
tampered = agent_answer.model_copy(deep=True)
figure = next(figure for figure in tampered.figures if figure.value is not None)
print(f"Changing '{figure.label}' from {figure.value} to {figure.value + 100}")
figure.value += 100

report = verify_answer(tampered, tool_log)

print("Verified:", report.passed)
pd.DataFrame([check.model_dump() for check in report.checks if not check.passed])

Caught. A reader would never spot a figure that is off by 100. The verifier does, because no tool returned that number.

This is how the BRD's first success criterion is enforced: *every figure in an answer can be traced to a stored value; none is produced by the AI model.*

## Step 4: From notebooks to production

In real work, a notebook is where you figure things out. Once it works, you rebuild it properly: split into files, with types, error handling, and tests. That rebuilt code is in this milestone's `retail_profit_stack/` folder.

| You built this in the notebooks | Production version | What changed |
|---|---|---|
| Loading tables with pandas (notebook 1) | `store.py` | Loads once into DuckDB and checks each table against its contract |
| `AnalystAnswer` form (notebook 3) | `schemas.py` | The same form, plus typed shapes for every tool result |
| The six tools (notebooks 4 and 5) | `tools.py` | The same rules, in SQL, returning typed results |
| The agent and its prompt (notebook 5) | `agent.py` | One function, `run_analyst`, returns the answer, the tool log, and the verification |
| Comparing with the answer key (notebooks 3 and 5) | `evaluation.py` | Runs a set of questions with known answers and applies every verifier check |
| The verifier (this notebook) | `verify.py` | You have been using the production version already |

The production code must give the same results as the notebook code. Let's check two of the tools.

In [ ]:
notebook_result = lab_helpers.query_profit(LAST_MONTH, "category")["rows"]
production_result = production_tools.query_profit(LAST_MONTH, "category").model_dump()["rows"]
print("Category totals match:", notebook_result == production_result)

notebook_result = lab_helpers.compare_profit(LAST_MONTH, PREVIOUS_MONTH, "product")["rows"]
production_result = production_tools.compare_profit(LAST_MONTH, PREVIOUS_MONTH, "product").model_dump()["rows"]
print("Product changes match:", notebook_result == production_result)

A test in `tests/test_notebook_parity.py` makes this same comparison for all six tools, every time the tests run.

## Step 5: The evaluation set

One question isn't enough to trust a system. The production code has a small **evaluation set**: seven questions with known correct answers, including two that the analyst must decline.

In [ ]:
cases = load_cases()

pd.set_option("display.max_colwidth", None)
pd.DataFrame([{"case": case.case_id, "question": case.question, "expected": case.expected_status, "tests": case.tests} for case in cases])

Running it uses the production agent, `run_analyst`. That is everything from notebook 5 in a single function call.

This costs model requests: about 15 for the three "tiny" cases. It is switched off by default. Set `RUN_EVALUATION = True` to run it when you have requests to spare.

In [ ]:
RUN_EVALUATION = False

if RUN_EVALUATION:
    scores = run_evaluation(run_analyst, load_cases(tiny=True))
    for score in scores:
        failed = [check.name for check in score.checks if not check.passed]
        print(f"{score.case_id}: {'passed' if score.passed else 'failed ' + str(failed)}")
        if score.error:
            print("   The model did not return an answer:", score.error)
    display(pd.Series(summarise(scores), name="passed").to_frame())
else:
    print("Evaluation not run. Set RUN_EVALUATION = True to run it.")

## Step 6: Run the tests

Production code comes with tests. Open a terminal in this milestone's folder (`src/m1_mocked_business_answer`) and run:

```
uv run pytest
```

The tests use a scripted fake model in place of a real one, so they need no key, cost nothing, and give the same result every time.

> ### Your assignment
>
> 1. **Try a specific model.** `openrouter/free` picks a different free model each time. In your `.env`, set `MODEL_NAME` to one free model (their ids end in `:free`) and re-run notebooks 3 and 5. Which model does best with the agent?
> 2. **Break a rule.** In notebook 5, delete rule 3 from the agent's instructions (the one about quality flags) and re-run. Then come back here and run the verifier. Which checks fail? Put the rule back afterwards.
> 3. **Ask your own question.** In notebook 5, replace `MAIN_QUESTION` with a question of your own, such as "What was net sales for Apparel in September?" Work out the right answer with pandas first, as in notebook 1.

> ### What you built
>
> You took a business question from a requirements document to a checked answer:
>
> 1. You answered it by hand, to know what "right" looks like.
> 2. You made your first LLM calls, and used AI to draft metric definitions.
> 3. You asked the question with all the data in the prompt, and saw the limits of that.
> 4. You built a tool, ran the agent loop by hand, and saw what LangChain does for you.
> 5. You gave an agent six tools and compared it with the single call.
> 6. You verified an answer with code alone, and connected the notebooks to production.
>
> The lesson to carry forward: **let the model do what it is good at (understanding questions, choosing what to look up, explaining), keep every number in code, and check the result with code.**
>
> Everything behind the tools is still mocked: the profit table was prepared by hand. In Milestone 2 we replace it with real calculations from orders, refunds, and costs, behind the same six tools.